# mini-llm-runner: Building Generative LLM from Scratch
**Author:** Vikash Mehta ([@vkshdev](https://github.com/vkshdev))  
**Project:** build-mini-LLM
---

### Pipeline Structure
1. **Phase 1 (Pre-training):** Foundational language modeling on `roneneldan/TinyStories` (160M tokens) with a 42M parameter (12-layer) Transformer and custom 8k Byte-Level BPE tokenizer, allocating 90% of parameter budget (37.76M) directly to core compute.
2. **Phase 2 (Conversational SFT):** Supervised instruction tuning on `tatsu-lab/alpaca` with ChatML formatting and causal next-token loss masking.
3. **Inference & Benchmarking:** Quantitative throughput testing (tokens/second) and qualitative generation inspection across both base and aligned checkpoints.

## 1. Checking Hardware & Runtime Environment

In [1]:
import sys
import torch

print(f"Environment: Python {sys.version.split()[0]} | PyTorch {torch.__version__}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} ({p.total_memory / (1024**3):.2f} GB VRAM, Compute {p.major}.{p.minor})")
else:
    print("GPU: None detected (CPU mode)")


Environment: Python 3.13.16 | PyTorch 2.11.0+cu130
GPU 0: Tesla T4 (14.56 GB VRAM, Compute 7.5)
GPU 1: Tesla T4 (14.56 GB VRAM, Compute 7.5)


## 2. Cloning Codebase & Setting Up Dependencies

In [2]:
import os

REPO_URL = "https://github.com/vkshdev/build-mini-LLM.git"
PROJECT_DIR = "/kaggle/working/build-mini-LLM"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

if not os.path.exists(PROJECT_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repository exists. Pulling latest commits...")
    !git -C {PROJECT_DIR} pull origin main

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("[OK] Environment prepared.")

Cloning repository from https://github.com/vkshdev/build-mini-LLM.git...
Cloning into '/kaggle/working/build-mini-LLM'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 85 (delta 38), reused 76 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 406.54 KiB | 11.62 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/kaggle/working/build-mini-LLM
[OK] Environment prepared.


## 3. Authenticating Weights & Biases Dashboard

In [3]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    my_wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=my_wandb_key)
    print("[OK] Authenticated with W&B via Kaggle Secrets.")
except Exception:
    print("Kaggle secret not detected. Falling back to W&B login:")
    wandb.login()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vkshdev (vkshdev-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[OK] Authenticated with W&B via Kaggle Secrets.


## 4. Phase 1: Pre-training Base 42M LLM on TinyStories
Pre-train the 42.1M parameter 12-layer decoder-only Transformer across160M tokens using FP16 mixed precision.

In [4]:
!python -m src.train \
    --data_dir /kaggle/working/build-mini-LLM/data \
    --checkpoint_dir /kaggle/working/checkpoints \
    --max_steps 10000 \
    --batch_size 32 \
    --lr 6e-4 \
    --resume True

Starting Pre-training (Phase 1) | Device: CUDA | Precision: float16
Fetching 'roneneldan/TinyStories' from Hugging Face...
README.md: 100%|███████████████████████████| 1.06k/1.06k [00:00<00:00, 3.48MB/s]

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:  |  0.00B       
data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:  | 29.8MB,  666k
data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes: ▏| 44.7MB, 2.90M
data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   8%| | 20.9MB
data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:  13%|▏| 33.4MB
data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes: ▎| 91.1MB, 5.69M
data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes: ▋|  177MB, 11.7M
data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes: ▊|  210MB, 16.5M
data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes: █|  248MB, 22.3M
data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file: 

## 5. Phase 2: Conversational Fine-Tuning (Chat SFT)
Align the pre-trained weights into an interactive assistant using ChatML formatting and causal next-token loss masking.

In [5]:
!python -m src.sft_train \
    --base_model /kaggle/working/checkpoints/best_pretrained_28m.pt \
    --data_dir /kaggle/working/build-mini-LLM/data \
    --output_dir /kaggle/working/checkpoints \
    --max_steps 1500 \
    --batch_size 16 \
    --lr 3e-5

Starting Chat SFT (Phase 2) | Checkpoint: /kaggle/working/checkpoints/best_pretrained_28m.pt
Pre-trained weights loaded.
Loading instruction dataset (alpaca)...
README.md: 100%|███████████████████████████| 7.47k/7.47k [00:00<00:00, 17.3MB/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes: ▌| 13.0MB,  320k
data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes: █| 24.1MB, 2.35M
data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file: 100%|█| 24.2MB
Generating train split: 100%|██| 52002/52002 [00:00<00:00, 384985.86 examples/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: vkshdev (vkshdev-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Track

## 6. Testing & Benchmarking Model Outputs
Evaluate generation quality and measure throughput (tokens/second) across both pre-trained story continuations and conversational instruction responses.

In [6]:
import os
import time
import torch
from src.config import ModelConfig
from src.tokenizer import get_tokenizer
from src.model import SmallLLM

device = "cuda" if torch.cuda.is_available() else "cpu"
tok_path = "/kaggle/working/build-mini-LLM/data/tokenizer.json"
tokenizer = get_tokenizer(tok_path if os.path.exists(tok_path) else None)
stop_ids = [tokenizer.im_end_id, tokenizer.eos_id]

# Part A: Evaluate Base Pre-trained Model (Story Generation)
pretrained_path = "/kaggle/working/checkpoints/best_pretrained_28m.pt"
if os.path.exists(pretrained_path):
    print(f"Evaluating Base Pre-trained Model: {pretrained_path}")
    ckpt = torch.load(pretrained_path, map_location=device)
    state_dict = ckpt.get("model_state_dict", ckpt)
    cfg = ModelConfig()
    if "config" in ckpt and "n_layer" in ckpt["config"]:
        cfg.n_layer = ckpt["config"]["n_layer"]
    if "transformer.wte.weight" in state_dict:
        cfg.vocab_size = state_dict["transformer.wte.weight"].shape[0]
    else:
        cfg.vocab_size = tokenizer.vocab_size
    base_model = SmallLLM(cfg).to(device)
    base_model.load_state_dict(state_dict)
    base_model.eval()

    story_prompts = [
        "Once upon a time in a sunny meadow,",
        "A little girl named Lily found a shiny silver key and"
    ]
    for sp in story_prompts:
        tokens = torch.tensor(tokenizer.encode(sp), dtype=torch.long, device=device).unsqueeze(0)
        t0 = time.time()
        out = base_model.generate(tokens, max_new_tokens=60, temperature=0.75, top_k=35, stop_id=stop_ids)
        dt = time.time() - t0
        new_tokens = out[0, tokens.size(1):].tolist()
        story = tokenizer.decode(new_tokens).replace("<|endoftext|>", "").strip()
        print(f"Prompt:      {sp}")
        print(f"Story:       {sp} {story}")
        print(f"Throughput:  {len(new_tokens)} tokens in {dt:.2f}s ({len(new_tokens)/max(dt,1e-4):.1f} tok/s)\n")

# Part B: Evaluate Conversational Chatbot (Instruction Following)
chat_path = "/kaggle/working/checkpoints/chat_model_28m.pt"
if os.path.exists(chat_path):
    print(f"Evaluating Conversational Chatbot: {chat_path}")
    ckpt = torch.load(chat_path, map_location=device)
    state_dict = ckpt.get("model_state_dict", ckpt)
    cfg = ModelConfig()
    if "config" in ckpt and "n_layer" in ckpt["config"]:
        cfg.n_layer = ckpt["config"]["n_layer"]
    if "transformer.wte.weight" in state_dict:
        cfg.vocab_size = state_dict["transformer.wte.weight"].shape[0]
    else:
        cfg.vocab_size = tokenizer.vocab_size
    chat_model = SmallLLM(cfg).to(device)
    chat_model.load_state_dict(state_dict)
    chat_model.eval()

    def ask_bot(user_msg: str, max_new_tokens: int = 70):
        prompt_str = f"<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"
        tokens = torch.tensor(tokenizer.encode(prompt_str), dtype=torch.long, device=device).unsqueeze(0)
        t0 = time.time()
        out = chat_model.generate(tokens, max_new_tokens=max_new_tokens, temperature=0.7, top_k=30, top_p=0.9, stop_id=stop_ids)
        dt = time.time() - t0
        new_tokens = out[0, tokens.size(1):].tolist()
        reply = tokenizer.extract_reply(tokenizer.decode(new_tokens))
        print(f"User:        {user_msg}")
        print(f"Assistant:   {reply}")
        print(f"Throughput:  {len(new_tokens)} tokens in {dt:.2f}s ({len(new_tokens)/max(dt,1e-4):.1f} tok/s)\n")

    chat_queries = [
        "Can you explain why the sky looks blue in simple words?",
        "Write a 2-sentence story about a friendly mouse sharing an apple.",
        "What should you do if you see a lost dog in the park?"
    ]
    for q in chat_queries:
        ask_bot(q)


Evaluating Base Pre-trained Model: /kaggle/working/checkpoints/best_pretrained_28m.pt
Prompt:      Once upon a time in a sunny meadow,
Story:       Once upon a time in a sunny meadow, there was a little bunny named Benny. Benny loved to eat carrots and play with his friends. One day, Benny's friend, a little bird named Billy, came to visit him. Billy asked Benny if he wanted to play a game of tag. Benny was excited and said yes.

Benny
Throughput:  60 tokens in 0.75s (80.3 tok/s)

Prompt:      A little girl named Lily found a shiny silver key and
Story:       A little girl named Lily found a shiny silver key and was very happy. She loved her shiny silver key and played with it every day. She would lock it with her friends and watch them play.

One day, Lily went to the park to play. She saw a boy named Tom playing with a big ball. Lily wanted to play too,
Throughput:  60 tokens in 0.33s (182.0 tok/s)

Evaluating Conversational Chatbot: /kaggle/working/checkpoints/chat_model_28m.pt
User

## 7. Reviewing Checkpoint Artifacts
Inspect the checkpoint file sizes in `/kaggle/working/checkpoints`.

In [7]:
import glob

my_ckpts = glob.glob("/kaggle/working/checkpoints/*.pt")
if my_ckpts:
    print("Checkpoint artifacts:")
    for p in sorted(my_ckpts):
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f"  {os.path.basename(p):<28} {size_mb:8.2f} MB")
else:
    print("No checkpoint files found.")

Checkpoint artifacts:
  best_pretrained_28m.pt          88.30 MB
  chat_model_28m.pt               88.30 MB
  checkpoint_latest.pt           481.74 MB
